In [1]:
#pip install transformers[torch] accelerate -U

In [2]:
#pip install transformers datasets accelerate sentence-transformers torch --upgrade

In [3]:
#pip install datasets


In [4]:
#pip show datasets

In [5]:
#pip install torch

In [6]:
#pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118

In [7]:
from transformers import AutoModelForCausalLM, AutoTokenizer, TrainingArguments, Trainer
from datasets import load_dataset
import torch

# Check if CUDA is available
device = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')

# Load pre-trained model
model = AutoModelForCausalLM.from_pretrained('gpt2').to(device)

# Load the tokenizer
tokenizer = AutoTokenizer.from_pretrained('gpt2', use_fast=True)

# Load dataset
train_dataset = load_dataset('text', data_files={'train': 'train.txt'}, split='train')

# Tokenize the dataset and create labels
def tokenize_function(examples):
    tokenized_output = tokenizer(examples['text'], truncation=True, max_length=128)
    # Filter out sequences that are empty after tokenization
    non_empty = [len(ids) > 0 for ids in tokenized_output['input_ids']]
    tokenized_output = {k: [v[i] for i in range(len(v)) if non_empty[i]] for k, v in tokenized_output.items()}
    # Labels are identical to input_ids for causal LM
    tokenized_output['labels'] = tokenized_output['input_ids'].copy()
    return tokenized_output

tokenized_datasets = train_dataset.map(tokenize_function, batched=True, remove_columns=['text'])

# Define training arguments with `remove_unused_columns=False`
training_args = TrainingArguments(
    output_dir='./fine_tuned_gpt2',
    overwrite_output_dir=True,
    num_train_epochs=3,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    evaluation_strategy='epoch',
    save_strategy='epoch',
    fp16=torch.cuda.is_available(),
    logging_steps=500,
    save_total_limit=2,
    load_best_model_at_end=True,
    remove_unused_columns=False  # Prevents removing 'labels' or other columns
)

# Initialize Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets,
)

# Start training
trainer.train()

# Save the fine-tuned model and tokenizer
trainer.save_model('./fine_tuned_gpt2')
tokenizer.save_pretrained('./fine_tuned_gpt2')


c:\Users\Jack\anaconda3\Lib\site-packages\transformers\tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


Map:   0%|          | 0/3315111 [00:00<?, ? examples/s]

c:\Users\Jack\anaconda3\Lib\site-packages\transformers\training_args.py:1525: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(


  0%|          | 0/2105217 [00:00<?, ?it/s]

{'loss': 2.6844, 'grad_norm': 6.3917036056518555, 'learning_rate': 4.9988267242759304e-05, 'epoch': 0.0}
{'loss': 2.4303, 'grad_norm': 5.063030242919922, 'learning_rate': 4.997639198239421e-05, 'epoch': 0.0}
{'loss': 2.3235, 'grad_norm': 8.758378028869629, 'learning_rate': 4.996451672202913e-05, 'epoch': 0.0}


KeyboardInterrupt: 